# NB02 — The independent census (ground truth)

**Goal.** Build the answer to Q1 **without consulting SemCat**, so that NB03's grade of SemCat
is not circular. The only inputs here are (a) the access-aware live table listing and (b) live
`DESCRIBE` plus value sampling. SemCat is read exactly once at the end, to measure its
staleness/coverage against the live listing — a table-level comparison that uses none of
SemCat's semantic labels.

**Discriminating query (run first).** The strand-facet recall test at full scale. Per
`RESEARCH_PLAN.md` this separates **H1** from **H2/H3** on recall, and auditing the handful of
labels SemCat does apply starts the **H2-vs-H3** precision split.

**Outputs.** `data/ground_truth_columns.csv`, `data/annotation_sources.csv` (the primary
deliverable), `data/row_counts_verified.csv`, `data/semcat_coverage_diff.json`.

In [1]:
import sys, os, re, json, time
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
import berdl_notebook_utils as bnu
from berdl_notebook_utils.setup_spark_session import get_spark_session
from semcat import SEMCAT
from census import CENSUS, EXCLUDED

spark = get_spark_session(app_name="acatlas_nb02")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60); pd.set_option("display.max_rows", 200)
os.makedirs("../data", exist_ok=True)

## 1. Live, access-aware table listing

`get_db_structure()` returns only what this account can read. Note it returns a JSON *string*,
not a dict — a small trap worth naming, since treating it as a dict fails silently in a
`for k in struct` loop (it iterates characters).

In [2]:
struct = bnu.get_db_structure()
if isinstance(struct, str):
    struct = json.loads(struct)
live_tables = sorted(f"{db}.{t}" for db, ts in struct.items() for t in ts)
print(f"accessible databases: {len(struct)}")
print(f"accessible tables   : {len(live_tables)}")
tenants = sorted({d.split('.')[0] for d in struct})
print(f"tenant prefixes     : {tenants}")
print()
pd.Series([t.split('.')[0] for t in live_tables]).value_counts().rename("tables").to_frame().pipe(print)

accessible databases: 104
accessible tables   : 826
tenant prefixes     : ['bervodata', 'culturebotai', 'globalusers', 'kbase', 'kescience', 'mamillerpa', 'my', 'nmdc', 'refdata']

              tables
refdata          226
kbase            192
kescience        158
globalusers       92
nmdc              77
bervodata         74
culturebotai       3
mamillerpa         2
my                 2


## 2. Live schema sweep

~0.1 s per table. This is the measurement that made a non-circular design affordable: the
independent ground truth costs about two minutes, so there was no reason to shortcut it by
reusing SemCat's column list.

In [3]:
rows, fails = [], []
t0 = time.time()
for fq in live_tables:
    try:
        desc = spark.sql(f"DESCRIBE {fq}").collect()
    except Exception as e:
        fails.append((fq, str(e)[:140])); continue
    in_part = False
    for r in desc:
        cn = (r["col_name"] or "").strip()
        if cn.startswith("#"):
            if "Partition" in cn: in_part = True
            continue
        if not cn: continue
        tenant, database = fq.split(".")[0], fq.split(".")[1]
        rows.append(dict(fq=fq, tenant=tenant, database=database, table_name=fq.split(".", 2)[2],
                         column_name=cn, type=(r["data_type"] or "").strip(),
                         comment=(r["comment"] or ""), is_partition=in_part))
gt = pd.DataFrame(rows)
print(f"swept {gt.fq.nunique()} tables, {len(gt)} columns, {len(fails)} failures in {time.time()-t0:.0f}s")
if fails:
    print()
    print("failures (all one cause -- a stale user-warehouse location):")
    for f, m in fails[:3]: print("  ", f, "->", m[:90])
    print(f"  ... {len(fails)} total across {len({f.rsplit('.',1)[0] for f,_ in fails})} database(s)")
gt.to_csv("../data/ground_truth_columns.csv", index=False)
json.dump(fails, open("../data/sweep_failures.json", "w"), indent=1)

swept 826 tables, 67587 columns, 0 failures in 54s


## 3. Discriminating query — strand recall at full scale

SemCat labels 6 columns `content_type='strand'`. How many strand columns actually exist in the
tables this account can read? The comparison is restricted to accessible tenants, because those
are the only ones where ground truth can be established at all.

Adjudication is explicit rather than regex-only, because `strand` is the facet where the English
word is genuinely ambiguous: **genomic strand** (which DNA strand a feature lies on) and
**library strandedness** (whether an RNA-seq protocol preserved strand) are different things
that share a name.

In [4]:
ln = gt.column_name.str.lower()
RX_STRANDISH = r"strand|orientation|^frame$|^phase$"
strandish = gt[ln.str.contains(RX_STRANDISH, regex=True)].copy()

# Explicit adjudication: is this GENOMIC strand?
LIBRARY_STRANDEDNESS = {"is_stranded", "stranded_orientation"}
READING_FRAME = {"frame", "phase", "cds_phase"}
def adjudicate(col):
    c = col.lower()
    if c in LIBRARY_STRANDEDNESS: return "library_strandedness"
    if c in READING_FRAME:        return "reading_frame"
    if c in ("strand", "orientation") or c.endswith("_strand"): return "genomic_strand"
    return "other"
strandish["kind"] = strandish.column_name.map(adjudicate)
print(strandish.kind.value_counts().to_string())
print()
print(strandish[["fq", "column_name", "type", "kind"]].to_string(index=False))

kind
genomic_strand          17
other                    8
reading_frame            4
library_strandedness     2

                                                   fq                                                                                                                                                                                                                                                                                                      column_name    type                 kind
        bervodata.chess.leaf_area_index_site_metadata                                                                                                                                                                                                                                                                     a_b_record_sensor_orientation_azimuth_degree  double                other
globalusers.carbon_source_phenotypes.gene_calls_table                                                         

In [5]:
truth_strand = strandish[strandish.kind == "genomic_strand"]
print(f"GROUND TRUTH: {len(truth_strand)} genomic-strand columns "
      f"in {truth_strand.fq.nunique()} accessible tables")

sc_strand = spark.sql(f'''
SELECT concat(tenant,'.',database,'.',table_name) AS fq, column_name, tenant
FROM {SEMCAT}.columns WHERE content_type = 'strand'
''').toPandas()
print(f"SEMCAT LABELLED: {len(sc_strand)} columns as content_type='strand' (all tenants)")
print()
sc_strand["node"] = sc_strand.fq + "." + sc_strand.column_name
sc_strand["accessible"] = sc_strand.tenant.isin(tenants)
sc_strand["adjudicated"] = sc_strand.column_name.map(adjudicate)
print(sc_strand[["fq", "column_name", "accessible", "adjudicated"]].to_string(index=False))

GROUND TRUTH: 17 genomic-strand columns in 17 accessible tables


SEMCAT LABELLED: 6 columns as content_type='strand' (all tenants)

                                       fq          column_name  accessible          adjudicated
  enigma.genome_depot_enigma.browser_gene               strand       False       genomic_strand
enigma.genome_depot_enigma.browser_operon               strand       False       genomic_strand
kbase.nmdc_arkin.metatranscriptomics_gold               strand        True       genomic_strand
                    kescience.mgnify.gene               strand        True       genomic_strand
    nmdc.metadata.material_processing_set          is_stranded        True library_strandedness
    nmdc.metadata.material_processing_set stranded_orientation        True library_strandedness


In [6]:
# Recall and precision on the strand facet, restricted to accessible tenants.
truth_nodes = set(truth_strand.fq + "." + truth_strand.column_name)
sc_acc = sc_strand[sc_strand.accessible]
sc_nodes = set(sc_acc.node)

tp = len(sc_nodes & truth_nodes)
fp = len(sc_nodes - truth_nodes)
fn = len(truth_nodes - sc_nodes)
recall    = tp / (tp + fn) if (tp + fn) else float("nan")
precision = tp / (tp + fp) if (tp + fp) else float("nan")

print(f"accessible tenants only:")
print(f"  true positives  : {tp}")
print(f"  false positives : {fp}  {sorted(sc_nodes - truth_nodes)}")
print(f"  false negatives : {fn}")
print()
print(f"  RECALL    = {tp}/{tp+fn} = {recall:.1%}")
print(f"  PRECISION = {tp}/{tp+fp} = {precision:.1%}")
print()
print(f"H1 needs recall >= 80%        -> {'PASS' if recall >= 0.80 else 'FAIL'}")
print(f"H2 needs precision >= 80%     -> {'PASS' if precision >= 0.80 else 'FAIL'}")
print(f"H3 needs recall <50% AND precision <80% -> "
      f"{'both hold' if (recall < 0.50 and precision < 0.80) else 'does not hold'}")

accessible tenants only:
  true positives  : 2
  false positives : 2  ['nmdc.metadata.material_processing_set.is_stranded', 'nmdc.metadata.material_processing_set.stranded_orientation']
  false negatives : 15

  RECALL    = 2/17 = 11.8%
  PRECISION = 2/4 = 50.0%

H1 needs recall >= 80%        -> FAIL
H2 needs precision >= 80%     -> FAIL
H3 needs recall <50% AND precision <80% -> both hold


In [7]:
print("The columns SemCat missed:")
for n in sorted(truth_nodes - sc_nodes): print("  ", n)

The columns SemCat missed:
   globalusers.carbon_source_phenotypes.gene_calls_table.strand
   globalusers.kepangenome_parquet_1.feature.strand
   kbase.genomes.feature.strand
   kescience.fitnessbrowser.gene.strand
   kescience.mgnify.gene_crispr.orientation
   kescience.test_db.gene.strand
   mamillerpa.feature_model_demo.features.strand
   my.feature_model_demo.features.strand
   nmdc.metadata.genome_feature_set.strand
   refdata.bvbrc.bvbrc_feature_fact.strand
   refdata.fungal_db.pombase_cds_coords.strand
   refdata.fungal_db.pombase_exon_coords.strand
   refdata.fungal_db.pombase_gene_coords.strand
   refdata.jgi_mycocosm.features.strand
   refdata.jgi_virus.metavr_protein.strand


## 4. Facet classification over the full accessible catalog

Name-and-type based, deliberately independent of SemCat. This is a wide net for *candidate*
identification; it is not the final census, because name matching cannot tell a function
assignment from an ontology definition, nor a genomic span from a mass-spec scan number.
Section 5 adjudicates. The false-positive rate here is itself a finding — it is the reason
the census required reading schemas rather than trusting any regex, SemCat's or mine.

In [8]:
from semcat import RX_SEQREF
RX_CV = (r"pfam|interpro|^ipr$|^ipr_|_ipr$|tigrfam|panther|prosite|hamap|superfamily|gene3d|sfld|"
 r"pirsf|^ko$|^ko_|_ko$|^koid$|ko_term|kofam|kegg|^ec$|^ec_|_ec$|ec_number|enzyme_commission|"
 r"^go$|^go_|_go$|^goid$|go_term|go_id|goslim|^cog$|^cog_|_cog$|eggnog|ortholog|cazy|dbcan|"
 r"antismash|mibig|big_scape|^gcf|smcog|uniprot|swissprot|trembl|metacyc|rhea|reactome|"
 r"seed|subsystem|resfam|merops|tcdb|signature|^hmm|pathway|^module|ontology_terms|"
 r"protein_family|family_id|domain|motif|operon|regulon|^amr|virulence|specialty")
RX_SPAN2 = (r"^(start|end|stop|begin|beg)$|_(start|end|stop|begin|beg|pos)$|"
            r"^(start|end|stop|begin|beg)_|posfrom|posto|position_range")
RX_GEO2 = r"^lat$|^lon$|^long$|latitude|longitude|^lat_|_lat$|^lon_|_lon$|_long$|bbox|cent_lat|dec_lat|huc"
RX_PACKED = r"^location$|^locations$|^region$|^coord$|position_range"

gt["f_cv"]     = ln.str.contains(RX_CV, regex=True)
gt["f_span"]   = ln.str.contains(RX_SPAN2, regex=True) & ~ln.str.contains(RX_GEO2, regex=True)
gt["f_strand"] = gt.column_name.map(adjudicate).eq("genomic_strand")
gt["f_seqref"] = ln.str.contains(RX_SEQREF, regex=True)
gt["f_packed"] = ln.str.contains(RX_PACKED, regex=True)

tf = gt.groupby("fq")[["f_cv", "f_span", "f_strand", "f_seqref", "f_packed"]].max()
print("accessible tables carrying each facet:")
print(tf.sum().to_string())
print()
print(f"candidate tables (CV function OR span): {(tf.f_cv | tf.f_span).sum()}")
tf.to_csv("../data/ground_truth_table_facets.csv")

accessible tables carrying each facet:
f_cv        179
f_span       42
f_strand     17
f_seqref    243
f_packed     18

candidate tables (CV function OR span): 209


/tmp/ipykernel_178912/2479788294.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  gt["f_span"]   = ln.str.contains(RX_SPAN2, regex=True) & ~ln.str.contains(RX_GEO2, regex=True)


## 5. The census

The adjudicated answer, curated in `src/census.py`. Every entry was decided by reading the live
schema; where the schema was ambiguous the coordinate system was settled by sampling values, and
the `basis` field says which. Three value checks did real work:

- `nmdc.results.pfam_annotation_gff`: `end - start + 1 == alignment_length` exactly, so
  `start`/`end` are **amino-acid** positions — despite `_gff` in the name implying genome
  coordinates.
- `nmdc.results.annotation_kegg_orthology`: `q_end - q_start + 1 == aln_len`, so likewise
  amino-acid, and `s_start`/`s_end` are positions on a *different* protein (the KO reference).
- `kbase.content.c_<genome>_features`: `location` holds
  `[["NZ_QSIR01000001.1", 1, "+", 855]]` = `[contig, start, strand, length]`, and
  `ontology_terms` holds `{"SSO": {"SSO:000009137": [0, 0]}}`. Full coordinates and an ontology
  assignment, with no column name disclosing either.

In [9]:
cen = pd.DataFrame(CENSUS)
cen["id_systems_str"] = cen.id_systems.map(lambda xs: "; ".join(xs))
cen["n_id_systems"]   = cen.id_systems.map(len)
cen["span_str"] = cen.span.map(lambda s: "" if not s else (s[0] if not s[1] else f"{s[0]}..{s[1]}"))

counts = json.load(open("../data/row_counts_verified.json"))
cen["row_count"] = cen.fq.map(counts)

print(f"census entries: {len(cen)}")
print()
print("by category:");     print(cen.category.value_counts().to_string())
print()
print("by coordinate system:"); print(cen.coord_system.value_counts().to_string())
print()
print("by coordinate availability:"); print(cen.coords.value_counts().to_string())

census entries: 51

by category:
category
gene_function           22
both                    20
reference_vocabulary     5
region                   4

by coordinate system:
coord_system
none          27
nucleotide    15
amino_acid     5
multiple       4

by coordinate availability:
coords
no        27
yes       23
packed     1


In [10]:
# Region-annotation tables WITH usable coordinates -- the subset that matters for the
# BGC/collocation use case.
reg = cen[cen.category.isin(["both", "region"])].sort_values("coord_system")
print("REGION / SPAN-BEARING TABLES")
print(reg[["fq", "region_type", "coord_system", "coord_frame", "seq_ref", "span_str",
           "strand", "row_count"]].to_string(index=False))

REGION / SPAN-BEARING TABLES
                                                   fq               region_type coord_system                                                                                         coord_frame                            seq_ref                                        span_str         strand    row_count
              kbase.ke_pangenome.interproscan_domains      protein domain/motif   amino_acid                                                       pangenome gene-cluster representative protein                    gene_cluster_id                                     start..stop            NaN  833303130.0
                kbase.ke_pangenome.bakta_pfam_domains            protein domain   amino_acid                                                       pangenome gene-cluster representative protein                    gene_cluster_id                                     start..stop            NaN   18807208.0
                       kescience.interpro.protein2ipr      

In [11]:
print("GENE/PROTEIN FUNCTION ASSIGNMENT WITHOUT COORDINATES")
# NB: named fn_df, not fn -- `fn` is the strand false-negative COUNT from section 3, and
# shadowing it here silently corrupted the summary on the first run of this notebook.
fn_df = cen[cen.category == "gene_function"].sort_values("n_id_systems", ascending=False)
print(fn_df[["fq", "n_id_systems", "id_systems_str", "row_count"]].to_string(index=False))

GENE/PROTEIN FUNCTION ASSIGNMENT WITHOUT COORDINATES
                                                         fq  n_id_systems                                                                                                id_systems_str   row_count
               kbase.ke_pangenome.eggnog_mapper_annotations             9 eggNOG OGs; COG category; GO; EC; KEGG ko/Pathway/Module/Reaction/rclass/TC; BRITE; CAZy; BiGG Reaction; Pfam  93558330.0
                       kbase.ke_pangenome.bakta_annotations             9                       EC; GO; COG; COG category; KEGG KO; RefSeq; UniParc; UniRef100/90/50; free-text product 132538155.0
                               kescience.mgnify.gene_eggnog             7                                 eggNOG OGs; COG category; GO; EC; KEGG ko/pathway/module/reaction; CAZy; Pfam 144775367.0
                   kbase.ke_pangenome.interproscan_pathways             4                                                          KEGG; MetaCyc; Reactome; UniPath

In [12]:
# Identifier-system coverage across the census.
from collections import Counter
sysc = Counter()
for xs in cen.id_systems: 
    for x in xs: sysc[x.split(" (")[0].strip()] += 1
print("identifier systems, by number of tables carrying them:")
for k, v in sysc.most_common(40): print(f"  {v:>3}  {k}")

identifier systems, by number of tables carrying them:
   10  Pfam
    7  EC
    6  GO
    5  InterPro
    5  UniProt
    4  KEGG KO
    3  PDB
    3  free-text product
    3  COG category
    2  antiSMASH
    2  eggNOG OGs
    2  CAZy
    2  KEGG
    2  MetaCyc
    1  Gene3D
    1  SUPERFAMILY
    1  PANTHER
    1  PRINTS
    1  CDD
    1  NCBIfam
    1  ProSiteProfiles
    1  ProSitePatterns
    1  MobiDBLite
    1  SMART
    1  FunFam
    1  Coils
    1  Hamap
    1  PIRSF
    1  SFLD
    1  AntiFam
    1  and other InterPro member DB signatures
    1  TIGRFAM
    1  per-row via domainDb
    1  ligand IDs
    1  literature PMIDs
    1  MIBiG
    1  BiG-SCAPE
    1  BGC identifiers
    1  host/plant metadata
    1  Cas protein typing


In [13]:
# InterProScan's member databases: one table, many identifier systems. Enumerated from
# values, because no column name lists them.
mdb = spark.sql('''
SELECT analysis, COUNT(*) AS n FROM kbase.ke_pangenome.interproscan_domains
GROUP BY analysis ORDER BY n DESC''').toPandas()
print("kbase.ke_pangenome.interproscan_domains -- signature databases present:")
print(mdb.to_string(index=False))
print()
print(f"{len(mdb)} distinct analyses; note the 1-row 'Gene3' value, which looks like a "
      f"truncated 'Gene3D'.")
mdb.to_csv("../data/interproscan_member_dbs.csv", index=False)

kbase.ke_pangenome.interproscan_domains -- signature databases present:
       analysis         n
           Pfam 145825639
         Gene3D 141491686
    SUPERFAMILY 111637513
        PANTHER  77122154
         PRINTS  60868454
            CDD  47270978
        NCBIfam  46833159
ProSiteProfiles  41691626
     MobiDBLite  40472637
          SMART  31451092
         FunFam  26276159
ProSitePatterns  20338067
          Coils  14928529
          Hamap  13341361
          PIRSF  10375633
           SFLD   3218132
        AntiFam    160310
          Gene3         1

18 distinct analyses; note the 1-row 'Gene3' value, which looks like a truncated 'Gene3D'.


In [14]:
# Excluded: name-based false positives. Part of the honest answer.
print("EXCLUDED after inspection (name-based rules flagged these; they are not")
print("gene/protein function or region annotation):")
print()
for fq, why in EXCLUDED: print(f"  {fq}\n      {why}\n")

EXCLUDED after inspection (name-based rules flagged these; they are not
gene/protein function or region annotation):

  kbase.nmdc_arkin.metabolomics_gold
      mass spectrometry; 'start'/scan/retention-time columns, 'kegg' refers to compound not gene

  kbase.nmdc_arkin.lipidomics_gold
      mass spectrometry lipid features, not gene annotation

  kbase.nmdc_arkin.biochemical_features_metadata
      mass-spec feature metadata; all columns string-typed

  nmdc.metadata.material_processing_set
      laboratory protocol metadata; `stranded_orientation` and `is_stranded` are RNA-seq LIBRARY strandedness, not genomic strand -- and these are two of the six columns SemCat labels `strand`

  nmdc.metadata.workflow_execution_set
      assembly/workflow statistics; `contigs` is a count

  nmdc.metadata.biosample_set
      sample metadata; `location` is geographic

  globalusers.heterotrophicbacteria.whole_table
      12,640 columns of genome-level trait predictions; no gene-level assignment

  

In [15]:
# Emit the primary deliverable.
out = cen[["fq", "category", "region_type", "coords", "coord_system", "coord_frame",
           "seq_ref", "span_str", "strand", "id_systems_str", "n_id_systems",
           "row_count", "basis", "notes"]].rename(columns={"span_str": "span",
                                                           "id_systems_str": "id_systems"})
out = out.sort_values(["category", "coord_system", "fq"])
out.to_csv("../data/annotation_sources.csv", index=False)
print("wrote ../data/annotation_sources.csv", out.shape)
pd.DataFrame(EXCLUDED, columns=["fq", "reason"]).to_csv("../data/excluded_tables.csv", index=False)
pd.DataFrame([{"fq": k, "live_row_count": v} for k, v in counts.items()]).to_csv(
    "../data/row_counts_verified.csv", index=False)
print("wrote ../data/excluded_tables.csv, ../data/row_counts_verified.csv")

wrote ../data/annotation_sources.csv (51, 14)
wrote ../data/excluded_tables.csv, ../data/row_counts_verified.csv


## 6. SemCat coverage vs the live catalog

Table-level only — no semantic labels involved, so this stays independent of NB03's grade. It
answers a different question from precision/recall: does SemCat even know which tables exist?

In [16]:
sc_all = spark.sql(f'''
SELECT concat(tenant,'.',database,'.',table_name) AS fq, tenant, row_count
FROM {SEMCAT}.tables''').toPandas()
sc_in_acc = sc_all[sc_all.tenant.isin(tenants)]
live_set, sc_set = set(live_tables), set(sc_in_acc.fq)

stale   = sorted(sc_set - live_set)
missing = sorted(live_set - sc_set)
print(f"live accessible tables          : {len(live_set)}")
print(f"SemCat tables in those tenants  : {len(sc_set)}")
print(f"in SemCat but GONE live (stale) : {len(stale)}")
print(f"live but MISSING from SemCat    : {len(missing)}")
print(f"agreement                       : {len(sc_set & live_set)}/{len(live_set)} = "
      f"{len(sc_set & live_set)/len(live_set):.0%}")
json.dump({"stale": stale, "missing": missing}, open("../data/semcat_coverage_diff.json", "w"), indent=1)

live accessible tables          : 826
SemCat tables in those tenants  : 668
in SemCat but GONE live (stale) : 5
live but MISSING from SemCat    : 163
agreement                       : 663/826 = 80%


In [17]:
from collections import Counter
print("STALE -- in SemCat, no longer in the lakehouse (by database):")
for k, v in Counter(".".join(x.split(".")[:2]) for x in stale).most_common(10):
    print(f"  {v:>4}  {k}")
print()
print("MISSING -- live but unknown to SemCat (by database):")
for k, v in Counter(".".join(x.split(".")[:2]) for x in missing).most_common(14):
    print(f"  {v:>4}  {k}")

STALE -- in SemCat, no longer in the lakehouse (by database):
     5  refdata.gbif_backbone

MISSING -- live but unknown to SemCat (by database):
    25  bervodata.fao_soils
    25  bervodata.hwsd2
    25  globalusers.semantic_catalog
    13  bervodata.chess
     9  globalusers.genomes_test1
     9  globalusers.kepangenome_parquet_1
     8  bervodata.watershed_sfa_soil_moisture
     7  globalusers.phenotype_ontology_1
     7  globalusers.phenotype_parquet_1
     5  globalusers.aisynbio_test_1
     3  bervodata.bervo
     3  culturebotai.nometatraits_20260503
     3  globalusers.demo_test
     3  globalusers.ontology2


In [18]:
# Does the staleness/absence touch the census itself?
cen_fqs = set(cen.fq)
print("census tables absent from SemCat:")
for f in sorted(cen_fqs & set(missing)): print("  ", f, f"({counts.get(f):,} rows)" if counts.get(f) else "")
print()
# Verify the one empty table claim, since it is a strong statement.
for t in ["nmdc.metadata.genome_feature_set"]:
    n = spark.sql(f"SELECT COUNT(*) c FROM {t}").collect()[0][0]
    print(f"{t}: {n} rows  <- complete GFF3 schema (seqid/start/end/strand/phase/"
          f"feature_category_term_id) but no data")

census tables absent from SemCat:

nmdc.metadata.genome_feature_set: 0 rows  <- complete GFF3 schema (seqid/start/end/strand/phase/feature_category_term_id) but no data


### 6a. Is `tables.row_count` trustworthy?

A separate question from label quality, and the one place SemCat might be relied on without
verification. Compared against live Iceberg `COUNT(*)` for every census table. (This check
previously lived in an ad-hoc script; it is reproduced here so the figure quoted in
`REPORT.md` has notebook provenance.)

In [19]:
# Live Iceberg COUNT(*) for every census table, computed HERE rather than read from a
# cached file. An earlier version of this cell loaded data/row_counts_verified.json, which
# nothing in the chain regenerated -- so it silently carried a 2026-10-02 snapshot into a
# 2026-10-06 run and claimed to be "live". Caught in REVIEW_2.md. Counts are metadata-served
# by Iceberg, so this is seconds even at billion-row scale; there is no reason to cache it.
import time

targets = sorted({f for f in cen.fq if "<genome>" not in f})
# The kbase.content census entry is a per-genome table family; resolve it to the concrete
# tables actually present so the count is real rather than notional.
content_tbls = sorted({t for t in live_tables
                       if t.startswith("kbase.content.") and t.endswith(("_features", "_cdss"))})
targets = targets + content_tbls
print(f"counting {len(targets)} tables live ...")

t0 = time.time()
counts_live, count_fail = {}, {}
for i, fq in enumerate(targets, 1):
    try:
        counts_live[fq] = spark.sql(f"SELECT COUNT(*) AS c FROM {fq}").collect()[0][0]
    except Exception as e:
        counts_live[fq] = None
        count_fail[fq] = str(e)[:120]
    if i % 20 == 0:
        print(f"  {i}/{len(targets)}  {time.time()-t0:.0f}s")
print(f"done in {time.time()-t0:.0f}s; {len(count_fail)} failures")
for fq, msg in count_fail.items():
    print("  FAIL", fq, "->", msg[:90])

json.dump(counts_live, open("../data/row_counts_verified.json", "w"), indent=1)
pd.DataFrame([{"fq": k, "live_row_count": v} for k, v in sorted(counts_live.items())]).to_csv(
    "../data/row_counts_verified.csv", index=False)

sc_rc = dict(zip(sc_all.fq, sc_all.row_count))
rc_rows = []
for fq, live_n in counts_live.items():
    sc_n = sc_rc.get(fq)
    if fq not in sc_rc:
        status = "absent_from_semcat"
    elif pd.isna(sc_n):
        status = "semcat_null"
    elif live_n is None:
        status = "live_count_failed"
    elif int(sc_n) == int(live_n):
        status = "exact"
    else:
        status = "mismatch"
    rc_rows.append(dict(fq=fq, live=live_n, semcat=sc_n, status=status))
rc = pd.DataFrame(rc_rows)
print()
print(rc.status.value_counts().to_string())
print()
verifiable = rc[rc.status.isin(["exact", "mismatch"])]
exact_frac = (verifiable.status == "exact").mean()
print(f"verifiable (in SemCat with a non-null count, live count succeeded): {len(verifiable)}")
print(f"  exact    : {(verifiable.status=='exact').sum()}")
print(f"  mismatch : {(verifiable.status=='mismatch').sum()}")
print(f"  ACCURACY : {(verifiable.status=='exact').sum()}/{len(verifiable)} = {exact_frac:.1%}")
print()
print(f"H3 clause required row_count exact on >=95% of verifiable tables -> "
      f"{'MET' if exact_frac >= 0.95 else 'NOT MET'}")
print()
print("non-exact rows:")
print(rc[rc.status != "exact"].to_string(index=False))
rc.to_csv("../data/row_count_grade.csv", index=False)
print()
print("wrote ../data/row_counts_verified.json, row_counts_verified.csv, row_count_grade.csv")

# The census was emitted earlier in this notebook with row counts carried in src/census.py's
# notes; attach the freshly measured live counts and re-emit so annotation_sources.csv and
# these counts can never disagree.
cen["row_count"] = cen.fq.map(counts_live)
out2 = cen[["fq", "category", "region_type", "coords", "coord_system", "coord_frame",
            "seq_ref", "span_str", "strand", "id_systems_str", "n_id_systems",
            "row_count", "basis", "notes"]].rename(
    columns={"span_str": "span", "id_systems_str": "id_systems"})
out2.sort_values(["category", "coord_system", "fq"]).to_csv(
    "../data/annotation_sources.csv", index=False)
print("re-emitted ../data/annotation_sources.csv with live row counts",
      out2.shape, f"({out2.row_count.notna().sum()} with a count)")

counting 60 tables live ...


  20/60  3s


  40/60  6s


  60/60  9s
done in 9s; 0 failures

status
exact       59
mismatch     1

verifiable (in SemCat with a non-null count, live count succeeded): 60
  exact    : 59
  mismatch : 1
  ACCURACY : 59/60 = 98.3%

H3 clause required row_count exact on >=95% of verifiable tables -> MET

non-exact rows:
                              fq      live    semcat   status
refdata.jgi_virus.metavr_protein 369281941 369281934 mismatch

wrote ../data/row_counts_verified.json, row_counts_verified.csv, row_count_grade.csv
re-emitted ../data/annotation_sources.csv with live row counts (51, 14) (50 with a count)


In [20]:
summary = dict(
    accessible_databases=len(struct), accessible_tables=len(live_tables),
    columns_swept=int(len(gt)), sweep_failures=len(fails),
    strand_truth_columns=int(len(truth_strand)),
    strand_semcat_labelled_all_tenants=int(len(sc_strand)),
    strand_semcat_labelled_accessible=int(len(sc_acc)),
    strand_tp=tp, strand_fp=fp, strand_fn=fn,
    strand_recall=float(recall), strand_precision=float(precision),
    census_entries=int(len(cen)),
    census_with_coords=int((cen.coords != "no").sum()),
    census_nucleotide=int((cen.coord_system == "nucleotide").sum()),
    census_amino_acid=int((cen.coord_system == "amino_acid").sum()),
    census_multiple_frames=int((cen.coord_system == "multiple").sum()),
    excluded_false_positives=len(EXCLUDED),
    semcat_stale_tables=len(stale), semcat_missing_tables=len(missing),
    semcat_table_agreement=float(len(sc_set & live_set) / len(live_set)),
    row_count_verifiable=int(len(verifiable)),
    row_count_exact=int((verifiable.status == "exact").sum()),
    row_count_accuracy=float(exact_frac),
    interproscan_member_dbs=int(len(mdb)),
)
bad = {k: type(v).__name__ for k, v in summary.items()
       if not isinstance(v, (int, float, str, bool, type(None)))}
if bad: print("NON-SERIALIZABLE:", bad)
json.dump(summary, open("../data/nb02_summary.json", "w"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))

{
  "accessible_databases": 104,
  "accessible_tables": 826,
  "columns_swept": 67587,
  "sweep_failures": 0,
  "strand_truth_columns": 17,
  "strand_semcat_labelled_all_tenants": 6,
  "strand_semcat_labelled_accessible": 4,
  "strand_tp": 2,
  "strand_fp": 2,
  "strand_fn": 15,
  "strand_recall": 0.11764705882352941,
  "strand_precision": 0.5,
  "census_entries": 51,
  "census_with_coords": 24,
  "census_nucleotide": 15,
  "census_amino_acid": 5,
  "census_multiple_frames": 4,
  "excluded_false_positives": 13,
  "semcat_stale_tables": 5,
  "semcat_missing_tables": 163,
  "semcat_table_agreement": 0.8026634382566586,
  "row_count_verifiable": 60,
  "row_count_exact": 59,
  "row_count_accuracy": 0.9833333333333333,
  "interproscan_member_dbs": 18
}


## 9. Executed value check — `pfam_annotation_gff` coordinate system

The census row for `nmdc.results.pfam_annotation_gff` (`src/census.py`) records a value check —
`end - start + 1 == alignment_length` — that settled its `start`/`end` as **amino-acid**
positions despite the `_gff` name. That check was run interactively during exploration and
never saved. This cell re-runs it reproducibly: a seeded Bernoulli row sample across the whole
table, with N stated and the mismatch count saved to `data/pfam_coordinate_check.json`.

In [21]:
# Seeded random row sample over the full 2.68B-row table (not a LIMIT, which would read
# only the first files). Rows with a NULL in any of the three columns are counted separately
# and excluded from N, so N is exactly the number of rows the identity was tested on.
PFAM = "nmdc.results.pfam_annotation_gff"
FRACTION, SEED = 1e-5, 42

samp = (spark.table(PFAM)
        .select("gene_id", "pfam_accession", "start", "end", "alignment_length",
                "model_start", "model_end")
        .sample(withReplacement=False, fraction=FRACTION, seed=SEED)
        .cache())
samp.createOrReplaceTempView("pfam_sample")

agg = spark.sql("""
SELECT COUNT(*)                                                         AS sampled_rows,
       SUM(CASE WHEN start IS NULL OR `end` IS NULL OR alignment_length IS NULL
                THEN 1 ELSE 0 END)                                      AS null_rows,
       SUM(CASE WHEN start IS NOT NULL AND `end` IS NOT NULL AND alignment_length IS NOT NULL
                THEN 1 ELSE 0 END)                                      AS n_tested,
       SUM(CASE WHEN `end` - start + 1 != alignment_length THEN 1 ELSE 0 END) AS n_mismatch,
       MAX(`end`)                                                       AS max_end,
       MAX(model_end)                                                   AS max_model_end
FROM pfam_sample
""").collect()[0].asDict()

mismatch_examples = (spark.sql("""
SELECT gene_id, pfam_accession, start, `end`, alignment_length, model_start, model_end
FROM pfam_sample WHERE `end` - start + 1 != alignment_length LIMIT 20
""").toPandas())

pfam_check = dict(table=PFAM, method="DataFrame.sample (Bernoulli, without replacement)",
                  fraction=FRACTION, seed=SEED,
                  sampled_rows=int(agg["sampled_rows"]), null_rows=int(agg["null_rows"] or 0),
                  N=int(agg["n_tested"] or 0), mismatch_count=int(agg["n_mismatch"] or 0),
                  max_end=int(agg["max_end"]), max_model_end=int(agg["max_model_end"]),
                  mismatch_examples=mismatch_examples.to_dict(orient="records"))
json.dump(pfam_check, open("../data/pfam_coordinate_check.json", "w"), indent=1, default=str)
samp.unpersist()

print(f"sampled rows        : {pfam_check['sampled_rows']:,}  (fraction {FRACTION}, seed {SEED})")
print(f"rows with a NULL    : {pfam_check['null_rows']:,}  (excluded)")
print(f"N tested            : {pfam_check['N']:,}")
print(f"end-start+1 != alignment_length : {pfam_check['mismatch_count']:,}")
print(f"max(end) in sample  : {pfam_check['max_end']:,}   max(model_end): {pfam_check['max_model_end']:,}")
print()
print("IDENTITY HOLDS ON EVERY TESTED ROW" if pfam_check["mismatch_count"] == 0
      else f"IDENTITY FAILS ON {pfam_check['mismatch_count']} ROWS -- see mismatch_examples")
print("wrote ../data/pfam_coordinate_check.json")


sampled rows        : 26,779  (fraction 1e-05, seed 42)
rows with a NULL    : 0  (excluded)
N tested            : 26,779
end-start+1 != alignment_length : 0
max(end) in sample  : 3,625   max(model_end): 1,530

IDENTITY HOLDS ON EVERY TESTED ROW
wrote ../data/pfam_coordinate_check.json


## 10. Cross-check — SemCat's `coordinate` columns against this census

NB01 typed SemCat's 78 `content_type='coordinate'` columns using SemCat metadata alone
(`data/semcat_coordinate_typing.csv`). This census settled coordinate systems from live schema
and values without consulting SemCat. Where the two overlap, agreement is an independent second
evidence stream for the claim that one label covers several coordinate systems.

In [22]:
typing = pd.read_csv("../data/semcat_coordinate_typing.csv")
typing["fq"] = typing.tenant + "." + typing.database + "." + typing.table_name
xchk = typing.merge(cen[["fq", "coord_system"]], on="fq", how="inner")

# A SemCat-side inference contradicts the census if it names one sequence system and the
# census names a different single one ("multiple" is compatible with any molecular system).
def contradicts(inferred, census):
    if inferred in ("ambiguous", "geographic") or census == "multiple":
        return False
    if inferred == "hmm_model":
        return True          # HMM coordinates only exist in "multiple"-frame tables
    return inferred != census

xchk["contradiction"] = [contradicts(i, c) for i, c in zip(xchk.inferred_system, xchk.coord_system)]
print(f"SemCat 'coordinate' columns: {len(typing)}; in census tables: {len(xchk)}")
print()
print(pd.crosstab(xchk.inferred_system, xchk.coord_system).to_string())
print()
print(f"contradictions: {int(xchk.contradiction.sum())}")
print("census-confirmed systems under the one label:",
      sorted(set(xchk.loc[~xchk.inferred_system.isin(['ambiguous']), 'inferred_system'])))
xchk.to_csv("../data/semcat_coordinate_crosscheck.csv", index=False)
print("wrote ../data/semcat_coordinate_crosscheck.csv")


SemCat 'coordinate' columns: 78; in census tables: 36

coord_system     amino_acid  multiple  nucleotide
inferred_system                                  
ambiguous                 4         4           6
amino_acid                6         4           0
hmm_model                 0         2           0
nucleotide                0         0          10

contradictions: 0
census-confirmed systems under the one label: ['amino_acid', 'hmm_model', 'nucleotide']
wrote ../data/semcat_coordinate_crosscheck.csv
